# Document Parsing Experiments
In this notebook, I will test different parsing strategies for extracting text from PDF and DOCX files.

Libraries to test:
- **pdfplumber** for PDFs
- **python-docx** for DOCX files

Goal is to accurately extract text while maintaining as much structure (like bullet points and newlines) as possible.

In [2]:
import os
import docx
from pathlib import Path

# Set up paths relative to the notebook
BASE_DIR = Path().resolve().parent
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
RESUMES_DIR = RAW_DATA_DIR / "resumes"
JDS_DIR = RAW_DATA_DIR / "jds"

print(f"Raw Data Directory: {RAW_DATA_DIR}")
print(f"Found {len(list(RESUMES_DIR.glob('*.*')))} resumes.")
print(f"Found {len(list(JDS_DIR.glob('*.*')))} JDs.")

Raw Data Directory: /Users/neehanth/Documents/Data Scientist/agentic_resume_screener_project/agentic_resume_screener/data/raw
Found 18 resumes.
Found 5 JDs.


In [2]:
# Lets print the file names to verify that files exists
print("--- Resumes ---")
for f in RESUMES_DIR.glob('*.*'):
    print(f"  {f.name}")
print("\n--- Job Descriptions ---")
for f in JDS_DIR.glob('*.*'):
    print(f"  {f.name}")

--- Resumes ---
  maya_srinivasan_res_aiml_c01.docx
  elena_martinez_res_aiml_c03.pdf
  ethan_caldwell_res_data_analyst_c10.pdf
  maya_thompson_res_backend_software_engineer_c08.pdf
  jordan_lee_res_data_scientist_c05.docx
  .DS_Store
  alexandra_patel_res_data_scientist_c04.pdf
  olivia_harper_res_data_analyst_c11.docx
  priya_sharma_res_LogisticsAnalyst_c14.pdf
  marcus_henderson_res_LogisticsAnalyst_c13.docx
  jordan_webb_res_LogisticsAnalyst_c15.pdf
  jordan_kim_res_aiml_c02.pdf
  adrian_chen_res_backend_software_engineer_c07.pdf
  marcus_nguyen_res_data_analyst_c12.pdf
  priya_raman_res_data_scientist_c06.pdf
  priya_raman_res_backend_software_engineer_c09.docx

--- Job Descriptions ---
  datascientist_jp_morgan_chase.pdf
  logistics_analyst_caterpillar.pdf
  backend_software_engineer_openai.pdf
  aiml_engineer_gap_international.pdf
  data_analyst_world_pay.pdf


## 1. Testing PDF Parsing
Let's grab one of the PDF resumes and extract its text using `pdfplumber`.

In [ ]:
# Select a sample PDF resume (Single colum)
sample_pdf_path = next(RESUMES_DIR.glob("*.pdf"))
print(f"Testing PDF parsing on: {sample_pdf_path.name}\n")

def parse_pdf(file_path):
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted_text = page.extract_text()
                if extracted_text:
                    text += extracted_text + "\n"
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Run the extraction
pdf_text = parse_pdf(sample_pdf_path)
print(pdf_text)

Testing PDF parsing on: elena_martinez_res_aiml_c03.pdf

Elena Martinez
Philadelphia, PA • elena.martinez.ds@gmail.com • (215) 555-0117 • github.com/elena-builds •
linkedin.com/in/elenamartinez-tech
Product-minded software and analytics professional transitioning into AI TRANSFERABLE FIT
engineering after 8 years of experience building data products, workflow
• Consulting-environment
automation tools, experimentation platforms, and internal decision-support
communication
systems. Strong background in Python, cloud deployment, distributed
• Enterprise workflow
systems, applied analytics, and cross-functional delivery. Known for turning
optimization
ambiguous business needs into scalable technical solutions, mentoring peers,
• Technical architecture +
and communicating clearly with senior stakeholders. Brings highly delivery
transferable engineering, research, and systems-thinking skills. • Mentoring and
stakeholder alignment
CORE STRENGTHS
Engineering & Systems Python, SQL, APIs, backen

In [17]:
# Select a sample PDF resume (two column format)
sample_pdf_path = RESUMES_DIR / "jordan_kim_res_aiml_c02.pdf"
print(f"Testing PDF parsing on: {sample_pdf_path.name}\n")

def parse_pdf(file_path):
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted_text = page.extract_text()
                if extracted_text:
                    text += extracted_text + "\n"
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Run the extraction
pdf_text = parse_pdf(sample_pdf_path)
print(pdf_text)

Testing PDF parsing on: jordan_kim_res_aiml_c02.pdf

Philadelphia, PA
jordan.kim.ml@gmail.com
Jordan Kim
(484) 555-0198
github.com/jordankim-ai
Junior AI Engineer • LLM Prototyping • Applied NLP
jordankim.dev |
linkedin.com/in/jordankim
FIT SNAPSHOT Professional Summary
Junior AI Engineer with a recent Master’s in Computer Science and hands-on
• Strong technical overlap experience building LLM-based applications, RAG pipelines, and prompt-driven
with the role tools in academic and internship settings. Strong Python and PyTorch background
• Hands-on with RAG, prompt with practical exposure to Hugging Face, LangChain, OpenAI APIs, and cloud
engineering, and APIs deployment. Comfortable working in fast-moving, research-oriented environments
• Gap: does not yet meet 2+ and translating business problems into working prototypes.
years post-master's
Education
experience
CORE TOOLS M.S., Computer Science | Northeastern University | 2025
Focus: Machine Learning, NLP, Applied Deep Learning
LLM: 

`Insights:`

The parsing for the two-column resume <span style="color:#fa4a31">failed</span> completely.

By default, `pdfplumber`'s `extract_text()` method reads a document exactly like a laser printer prints it: horizontally, from top to bottom, strictly by Y-coordinates. When it hit Jordan's resume, it saw "FIT SNAPSHOT" on the left and "Professional Summary" on the right at the exact same Y-coordinate. It didn't know they were in separate columns, so it mashed them together into: `FIT SNAPSHOT Professional Summary`.

If we pass this scrambled string to your LLM Extraction Node, it will fail to understand the candidate's history because the semantic context is destroyed.

In [20]:
# Lets use `pymupdf` instead of `pdfplumber`
import fitz  # PyMuPDF

# Select the two-column resume
sample_pdf_path = RESUMES_DIR / "jordan_kim_res_aiml_c02.pdf"

def parse_pdf_blocks(file_path):
    text = ""
    try:
        # Open the document
        doc = fitz.open(file_path)
        for page in doc:
            # Extract text as blocks (paragraphs/columns)
            blocks = page.get_text("blocks")
            
            # Sort blocks primarily by vertical position, then horizontal
            # PyMuPDF usually returns blocks in logical reading order automatically
            for block in blocks:
                block_text = block[4] # The actual text string is the 5th element
                text += block_text + "\n"
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Run the extraction
pdf_text = parse_pdf_blocks(sample_pdf_path)
print(pdf_text)

Jordan Kim

Junior AI Engineer  •  LLM Prototyping  •  Applied NLP

Philadelphia, PA

jordan.kim.ml@gmail.com

(484) 555-0198

github.com/jordankim-ai

jordankim.dev | 

linkedin.com/in/jordankim

FIT SNAPSHOT

• Strong technical overlap 

with the role

• Hands-on with RAG, prompt 

engineering, and APIs

• Gap: does not yet meet 2+ 

years post-master's 

experience

CORE TOOLS

LLM: OpenAI, Claude, LLaMA

Frameworks: Hugging Face, 

LangChain

ML: PyTorch, TensorFlow

Infra: AWS, Docker, FastAPI

Search: FAISS, ChromaDB

Code: Python, SQL, Git

COMMUNITY

• University research 

publication (2025)

• Open-source LangChain 

contribution

• Technical blogging on RAG 

and evaluation

Professional Summary

Junior AI Engineer with a recent Master’s in Computer Science and hands-on 
experience building LLM-based applications, RAG pipelines, and prompt-driven 
tools in academic and internship settings. Strong Python and PyTorch background 
with practical exposure to Hugging Face, LangCha

`Insights:`

It is a complete <span style="color:#009b01">success</span> . Looking at the output in the cell, `PyMuPDF` (`fitz`) parsed the document as logical blocks rather than horizontal stripes.
* It kept the "FIT SNAPSHOT" and the "CORE TOOLS" sections entirely intact.
* It then cleanly transitioned into the "Professional Summary" and "Experience" without interweaving the words.

## 2. Testing DOCX Parsing
Now let's grab one of the DOCX resumes and extract its text using `python-docx`.

In [ ]:
# Select a sample DOCX resume (Single column)
sample_docx_path = next(RESUMES_DIR.glob("*.docx"))
print(f"Testing DOCX parsing on: {sample_docx_path.name}\n")

def parse_docx(file_path):
    text = ""
    try:
        doc = docx.Document(file_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Run the extraction
docx_text = parse_docx(sample_docx_path)
print(docx_text)

Testing DOCX parsing on: maya_srinivasan_res_aiml_c01.docx

Dr. Maya Srinivasan
Philadelphia, PA  |  maya.srinivasan.ai@gmail.com  |  (267) 555-0142  |  github.com/mayasri-ai  |  mayasrinivasan.dev  |  linkedin.com/in/mayasrinivasan
PROFESSIONAL SUMMARY
AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, and deploying large language model systems for enterprise use cases. Strong background in retrieval-augmented generation (RAG), prompt engineering, model optimization, agentic workflows, and scalable cloud deployment. Experienced partnering with cross-functional stakeholders to translate ambiguous business challenges into production-ready AI systems. Active contributor to the AI community through publications, open-source tooling, and technical writing.
CORE SKILLS
EDUCATION
Ph.D., Computer Science | Carnegie Mellon University | 2020
Focus: Natural Language Processing, Transformer Architectures, Human-in-the-Loop AI
M.S., Computer S

In [22]:
# Select a sample DOCX resume (Two column)
sample_docx_path = RESUMES_DIR / "olivia_harper_res_data_analyst_c11.docx"
print(f"Testing DOCX parsing on: {sample_docx_path.name}\n")

def parse_docx(file_path):
    text = ""
    try:
        doc = docx.Document(file_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# Run the extraction
docx_text = parse_docx(sample_docx_path)
print(docx_text)

Testing DOCX parsing on: olivia_harper_res_data_analyst_c11.docx

Olivia Harper
Cincinnati, OH  •  olivia.harper.analytics@gmail.com  •  (859) 555-0191  •  github.com/oharper-analytics  •  linkedin.com/in/oliviaharper


`Insights:`

In Microsoft Word, the easiest way to create a multi-column layout is to use invisible tables. The standard `python-docx` library reads text by iterating through `document.paragraphs`. However, text inside a table is not considered a standard paragraph by the library - it is hidden inside `document.tables`.

Because the code only iterated through paragraphs, it completely skipped Maya's skills section and Olivia's entire resume!

To fix this, we need to write a parser that can read both standard paragraphs and iterate through every cell in every table.

In [34]:
# Updated parser for .docx files
def parse_docx(file_path):
    try:
        doc = docx.Document(file_path)
        full_text = []

        # 1. Extract standard paragraphs
        for para in doc.paragraphs:
            if para.text.strip():
                full_text.append(para.text.strip())

        # 2. Extract text hidden inside tables
        for table in doc.tables:
            for row in table.rows:
                row_data = []
                for cell in row.cells:
                    clean_cell = cell.text.strip()
                    if clean_cell:
                        # Avoid duplicating cells if they are merged
                        if clean_cell not in row_data: 
                            row_data.append(clean_cell)
                if row_data:
                    full_text.append("\n".join(row_data))

        return "\n".join(full_text)
    
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

In [36]:
# Test on the single-column DOCX with the hidden table (Maya Srinivasan)
maya_path = RESUMES_DIR / "maya_srinivasan_res_aiml_c01.docx"
print(f"Testing DOCX parsing on: {maya_path.name}\n")
print(parse_docx(maya_path))

Testing DOCX parsing on: maya_srinivasan_res_aiml_c01.docx

Dr. Maya Srinivasan
Philadelphia, PA  |  maya.srinivasan.ai@gmail.com  |  (267) 555-0142  |  github.com/mayasri-ai  |  mayasrinivasan.dev  |  linkedin.com/in/mayasrinivasan
PROFESSIONAL SUMMARY
AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, and deploying large language model systems for enterprise use cases. Strong background in retrieval-augmented generation (RAG), prompt engineering, model optimization, agentic workflows, and scalable cloud deployment. Experienced partnering with cross-functional stakeholders to translate ambiguous business challenges into production-ready AI systems. Active contributor to the AI community through publications, open-source tooling, and technical writing.
CORE SKILLS
EDUCATION
Ph.D., Computer Science | Carnegie Mellon University | 2020
Focus: Natural Language Processing, Transformer Architectures, Human-in-the-Loop AI
M.S., Computer S

`Insights:`

Because the code loops through all the paragraphs first, and all the tables second, it completely *destroyed the natural top-to-bottom reading order* of the document. It read Maya's "CORE SKILLS" header (a paragraph), skipped the table beneath it, read the rest of the resume, and then dumped all the tables at the very end.

Under the hood, `python-docx` exposes two completely separate lists: `doc.paragraphs` and `doc.tables`.

In [35]:
# Test on the failed two-column DOCX (Olivia Harper)
olivia_path = RESUMES_DIR / "olivia_harper_res_data_analyst_c11.docx"
print(f"Testing DOCX parsing on: {olivia_path.name}\n")
print(parse_docx(olivia_path))

Testing DOCX parsing on: olivia_harper_res_data_analyst_c11.docx

Olivia Harper
Cincinnati, OH  •  olivia.harper.analytics@gmail.com  •  (859) 555-0191  •  github.com/oharper-analytics  •  linkedin.com/in/oliviaharper
SUMMARY
Early-career data analyst with strong SQL and Python skills, hands-on experience with statistics coursework, dashboarding, and analytics internships, and a growing portfolio in fraud, product utilization, and marketing analysis.
SKILLS
SQL, Python, R, Excel
Pandas, NumPy, scikit-learn
Tableau, Power BI, Git
Sampling, QA, trend analysis
Relational databases
A/B test support
Data storytelling
PROJECTS
Fraud trend explorer - Built Python/SQL notebooks on public transaction-like data to classify anomaly patterns, investigate feature importance, and summarize signals for analyst review.
Utilization cohort dashboard - Created Tableau and SQL views to monitor user engagement by cohort, segment, and campaign exposure for mock product decision support.
Marketing readout te

`Fix:` Hierarchy issue

Instead of two seperate lists, read the underlying Word XML (oxml) in its true sequential order.

Look at the document `body` and say: "Give me the next element. Is it a paragraph? Extract it. Is it a table? Extract it.

In [6]:
import docx
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from docx.text.paragraph import Paragraph
from docx.table import Table

In [3]:
# Open the document
maya_path = RESUMES_DIR / "maya_srinivasan_res_aiml_c01.docx"
doc = docx.Document(maya_path)

print(doc)

In [4]:
# Grab the raw XML body (the container holding everything)
parent_elm = doc.element.body
print(f"Successfully grabbed the document body: {type(parent_elm)}")

Successfully grabbed the document body: <class 'docx.oxml.document.CT_Body'>


In [ ]:
# Iterate through the XML children one by one
print("INSPECTING THE FIRST 10 ELEMENTS IN ORDER:")
print("="*50)
for i, child in enumerate(parent_elm.iterchildren()):
    if i >= 10:
        break

    print(f"\n--- Item {i+1} ---")
    
    # Print the raw, ugly XML tag name (e.g., '{http://schemas.openxmlformats.org/...}p')
    print(f"Raw XML Tag: {child.tag.split('}')[-1]}") # Splitting to remove the long URL prefix

    # If the raw XML is a Paragraph tag
    if isinstance(child, CT_P):
        print("Detected Type: Paragraph")
        
        # Wrap the raw XML in a "Pretty" Python object
        pretty_para = Paragraph(child, doc)
        
        # Print a snippet of the actual text
        snippet = pretty_para.text.strip()[:100]
        if snippet:
            print(f"Extracted Text: '{snippet}...'")
        else:
            print("Extracted Text: [Blank Line]")

    # If the raw XML is a Table tag
    elif isinstance(child, CT_Tbl):
        print("Detected Type: Table")
        
        # Wrap the raw XML in our "Pretty" Python object
        pretty_table = Table(child, doc)
        print(f"Extracted Text: [Table object containing {len(pretty_table.rows)} rows]")

    # If it's something else (like a page break or formatting artifact)
    else:
        print("Detected Type: Other formatting/structural element")

INSPECTING THE FIRST 10 ELEMENTS IN ORDER:

--- Item 1 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'Dr. Maya Srinivasan...'

--- Item 2 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'Philadelphia, PA  |  maya.srinivasan.ai@gmail.com  |  (267) 555-0142  |  github.com/mayasri-ai  |  m...'

--- Item 3 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'PROFESSIONAL SUMMARY...'

--- Item 4 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evalua...'

--- Item 5 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'CORE SKILLS...'

--- Item 6 ---
Raw XML Tag: tbl
Detected Type: Table
Extracted Text: [Table object containing 7 rows]

--- Item 7 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'EDUCATION...'

--- Item 8 ---
Raw XML Tag: p
Detected Type: Paragraph
Extracted Text: 'Ph.D., Computer Science | Carnegie Mellon University |

In [12]:
# Updated parser with XML Traversal to maintain the Hierarchy
import docx
from docx.oxml.text.paragraph import CT_P
from docx.oxml.table import CT_Tbl
from docx.text.paragraph import Paragraph
from docx.table import Table

def iter_block_items(parent):
    """
    Yields each paragraph and table child within *parent*, in document order.
    This solves the python-docx hierarchy issue by reading the raw XML sequence.
    """
    if isinstance(parent, docx.document.Document):
        parent_elm = parent.element.body
    elif isinstance(parent, docx.table._Cell):
        parent_elm = parent._tc
    else:
        raise ValueError("Unsupported parent type")

    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)

def parse_docx_sequential(file_path):
    try:
        doc = docx.Document(file_path)
        full_text = []

        # Iterate through paragraphs and tables exactly as they appear
        for block in iter_block_items(doc):
            if isinstance(block, Paragraph):
                if block.text.strip():
                    full_text.append(block.text.strip())
            
            elif isinstance(block, Table):
                for row in block.rows:
                    row_data = []
                    for cell in row.cells:
                        clean_cell = cell.text.strip()
                        # Avoid empty cells and duplicated merged cells
                        if clean_cell and clean_cell not in row_data: 
                            row_data.append(clean_cell)
                    if row_data:
                        # Join row data with a clear delimiter for the LLM
                        full_text.append("\n".join(row_data))

        return "\n".join(full_text)
    
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

In [13]:
# Test on the single-column DOCX with the hidden table (Maya Srinivasan)
maya_path = RESUMES_DIR / "maya_srinivasan_res_aiml_c01.docx"
print(f"Testing Sequential DOCX parsing on: {maya_path.name}\n")
print(parse_docx_sequential(maya_path))

Testing Sequential DOCX parsing on: maya_srinivasan_res_aiml_c01.docx

Dr. Maya Srinivasan
Philadelphia, PA  |  maya.srinivasan.ai@gmail.com  |  (267) 555-0142  |  github.com/mayasri-ai  |  mayasrinivasan.dev  |  linkedin.com/in/mayasrinivasan
PROFESSIONAL SUMMARY
AI/LLM Engineer and applied NLP researcher with 5+ years of experience building, fine-tuning, evaluating, and deploying large language model systems for enterprise use cases. Strong background in retrieval-augmented generation (RAG), prompt engineering, model optimization, agentic workflows, and scalable cloud deployment. Experienced partnering with cross-functional stakeholders to translate ambiguous business challenges into production-ready AI systems. Active contributor to the AI community through publications, open-source tooling, and technical writing.
CORE SKILLS
LLM / GenAI
Fine-tuning, instruction tuning, prompt engineering, RAG, agent design, evaluation pipelines
Frameworks / APIs
Hugging Face, LangChain, LangGraph, 

In [14]:
# Test on the failed two-column DOCX (Olivia Harper)
olivia_path = RESUMES_DIR / "olivia_harper_res_data_analyst_c11.docx"
print(f"Testing DOCX parsing on: {olivia_path.name}\n")
print(parse_docx_sequential(olivia_path))

Testing DOCX parsing on: olivia_harper_res_data_analyst_c11.docx

Olivia Harper
Cincinnati, OH  •  olivia.harper.analytics@gmail.com  •  (859) 555-0191  •  github.com/oharper-analytics  •  linkedin.com/in/oliviaharper
SUMMARY
Early-career data analyst with strong SQL and Python skills, hands-on experience with statistics coursework, dashboarding, and analytics internships, and a growing portfolio in fraud, product utilization, and marketing analysis.
SKILLS
SQL, Python, R, Excel
Pandas, NumPy, scikit-learn
Tableau, Power BI, Git
Sampling, QA, trend analysis
Relational databases
A/B test support
Data storytelling
PROJECTS
Fraud trend explorer - Built Python/SQL notebooks on public transaction-like data to classify anomaly patterns, investigate feature importance, and summarize signals for analyst review.
Utilization cohort dashboard - Created Tableau and SQL views to monitor user engagement by cohort, segment, and campaign exposure for mock product decision support.
Marketing readout te

`Insights:`

Now the order is same as in the original document and also works with the two-column Word document.